# Ferienanspruch — OpenFisca Implementation

Transformation of a Swiss legal provision on vacation entitlement into OpenFisca code.

## Legal text

> 1 Die Angestellten haben pro Kalenderjahr Anspruch auf Ferien von:
> - a. 6 Wochen bis und mit dem Kalenderjahr, in dem sie das 20. Altersjahr vollenden;
> - b. 5 Wochen vom Beginn des Kalenderjahres an, in dem sie das 21. Altersjahr vollenden;
> - c. 6 Wochen vom Beginn des Kalenderjahres an, in dem sie das 50. Altersjahr vollenden;
> - d. 7 Wochen vom Beginn des Kalenderjahres an, in dem sie das 60. Altersjahr vollenden.

## Tiers

| Age reached in calendar year | Weeks |
|---|---|
| ≤ 20 | 6 |
| 21–49 | 5 |
| 50–59 | 6 |
| ≥ 60 | 7 |

## 1. Setup

Install OpenFisca (using the country template as a base since `openfisca-switzerland` does not yet exist as a published package).

In [ ]:
!pip install openfisca-core[web-api] openfisca-country-template

In [ ]:
import os
from pathlib import Path

# Working directory layout
BASE = Path("vacation_module")
PARAMS_DIR = BASE / "parameters" / "employment"
TESTS_DIR = BASE / "tests"

PARAMS_DIR.mkdir(parents=True, exist_ok=True)
TESTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Created layout under {BASE.resolve()}")

## 2. Variable class

The formula keys off the *calendar year in which the employee turns the threshold age*, not the age at a single instant — that's what the German phrase "vom Beginn des Kalenderjahres an, in dem sie das X. Altersjahr vollenden" means.

In [ ]:
variable_code = '''
from openfisca_core.model_api import *
from openfisca_country_template.entities import Person


class vacation_weeks_entitlement(Variable):
    value_type = float
    entity = Person
    definition_period = YEAR
    label = "Ferienanspruch in Wochen pro Kalenderjahr (altersabhaengig)"
    reference = "Art. 1 - Ferienanspruch der Angestellten"

    def formula(person, period, parameters):
        # Age reached during the calendar year = calendar year - birth year.
        # The law triggers the new tier from the *start* of the year the employee
        # turns the threshold age.
        date_of_birth = person("date_of_birth", period)
        birth_year = date_of_birth.astype("datetime64[Y]").astype(int) + 1970
        age_reached = period.start.year - birth_year

        p = parameters(period).employment.vacation

        return select(
            [
                age_reached <= p.age_threshold_youth,    # up to 20
                age_reached < p.age_threshold_middle,    # 21-49
                age_reached < p.age_threshold_senior,    # 50-59
                age_reached >= p.age_threshold_senior,   # 60+
            ],
            [
                p.weeks_youth,      # 6
                p.weeks_standard,   # 5
                p.weeks_middle,     # 6
                p.weeks_senior,     # 7
            ],
        )
'''

(BASE / "vacation_weeks_entitlement.py").write_text(variable_code)
print("Wrote vacation_weeks_entitlement.py")

## 3. Parameter YAML

Age thresholds and week values, dated so they can evolve over time.

In [ ]:
vacation_yaml = '''description: Altersabhaengiger Ferienanspruch der Angestellten
metadata:
  reference:
    - title: "Ferienanspruch der Angestellten (Art. 1)"

age_threshold_youth:
  description: Obere Altersgrenze fuer den erhoehten Ferienanspruch junger Angestellter (bis und mit diesem Altersjahr)
  metadata:
    unit: year
  values:
    2000-01-01:
      value: 20

age_threshold_middle:
  description: Altersjahr, ab dessen Kalenderjahr der mittlere Ferienanspruch (6 Wochen) gilt
  metadata:
    unit: year
  values:
    2000-01-01:
      value: 50

age_threshold_senior:
  description: Altersjahr, ab dessen Kalenderjahr der hoechste Ferienanspruch (7 Wochen) gilt
  metadata:
    unit: year
  values:
    2000-01-01:
      value: 60

weeks_youth:
  description: Ferien in Wochen bis zum Kalenderjahr, in dem das 20. Altersjahr vollendet wird
  values:
    2000-01-01:
      value: 6

weeks_standard:
  description: Ferien in Wochen ab dem Kalenderjahr, in dem das 21. Altersjahr vollendet wird
  values:
    2000-01-01:
      value: 5

weeks_middle:
  description: Ferien in Wochen ab dem Kalenderjahr, in dem das 50. Altersjahr vollendet wird
  values:
    2000-01-01:
      value: 6

weeks_senior:
  description: Ferien in Wochen ab dem Kalenderjahr, in dem das 60. Altersjahr vollendet wird
  values:
    2000-01-01:
      value: 7
'''

(PARAMS_DIR / "vacation.yaml").write_text(vacation_yaml)
print(f"Wrote {PARAMS_DIR / 'vacation.yaml'}")

## 4. YAML tests covering all four tiers

In [ ]:
test_yaml = '''- name: "Angestellter mit 19 Jahren - 6 Wochen Ferien"
  period: 2024
  input:
    date_of_birth: 2005-06-15
  output:
    vacation_weeks_entitlement: 6

- name: "Angestellter wird 20 im Kalenderjahr - letztes Jahr mit 6 Wochen"
  period: 2024
  input:
    date_of_birth: 2004-11-02
  output:
    vacation_weeks_entitlement: 6

- name: "Angestellter wird 21 im Kalenderjahr - 5 Wochen"
  period: 2024
  input:
    date_of_birth: 2003-03-10
  output:
    vacation_weeks_entitlement: 5

- name: "Angestellter 35 Jahre - 5 Wochen"
  period: 2024
  input:
    date_of_birth: 1989-07-01
  output:
    vacation_weeks_entitlement: 5

- name: "Angestellter wird 50 im Kalenderjahr - 6 Wochen"
  period: 2024
  input:
    date_of_birth: 1974-08-20
  output:
    vacation_weeks_entitlement: 6

- name: "Angestellter wird 60 im Kalenderjahr - 7 Wochen"
  period: 2024
  input:
    date_of_birth: 1964-12-31
  output:
    vacation_weeks_entitlement: 7
'''

(TESTS_DIR / "vacation_weeks_entitlement.yaml").write_text(test_yaml)
print(f"Wrote {TESTS_DIR / 'vacation_weeks_entitlement.yaml'}")

## 5. Wire the variable & parameter into a TaxBenefitSystem and run a quick simulation

In [ ]:
from openfisca_country_template import CountryTaxBenefitSystem
from openfisca_core.model_api import Variable, YEAR, select
from openfisca_core.parameters import load_parameter_file
from openfisca_country_template.entities import Person
import numpy as np


class vacation_weeks_entitlement(Variable):
    value_type = float
    entity = Person
    definition_period = YEAR
    label = "Ferienanspruch in Wochen pro Kalenderjahr (altersabhaengig)"
    reference = "Art. 1 - Ferienanspruch der Angestellten"

    def formula(person, period, parameters):
        date_of_birth = person("date_of_birth", period)
        birth_year = date_of_birth.astype("datetime64[Y]").astype(int) + 1970
        age_reached = period.start.year - birth_year

        p = parameters(period).employment.vacation

        return select(
            [
                age_reached <= p.age_threshold_youth,
                age_reached < p.age_threshold_middle,
                age_reached < p.age_threshold_senior,
                age_reached >= p.age_threshold_senior,
            ],
            [
                p.weeks_youth,
                p.weeks_standard,
                p.weeks_middle,
                p.weeks_senior,
            ],
        )


tax_benefit_system = CountryTaxBenefitSystem()

# Load and graft the vacation parameter node onto the existing tree.
vacation_node = load_parameter_file(
    str((PARAMS_DIR / "vacation.yaml").resolve()),
    name="vacation",
)
if not hasattr(tax_benefit_system.parameters, "employment"):
    from openfisca_core.parameters import ParameterNode
    employment = ParameterNode("employment", data={})
    tax_benefit_system.parameters.add_child("employment", employment)
tax_benefit_system.parameters.employment.add_child("vacation", vacation_node)

tax_benefit_system.add_variable(vacation_weeks_entitlement)
print("TaxBenefitSystem ready.")

In [ ]:
from openfisca_core.simulation_builder import SimulationBuilder

situation = {
    "persons": {
        "alice":   {"date_of_birth": {"ETERNITY": "2005-06-15"}},  # 19 in 2024 -> 6
        "bob":     {"date_of_birth": {"ETERNITY": "2003-03-10"}},  # 21 in 2024 -> 5
        "carol":   {"date_of_birth": {"ETERNITY": "1974-08-20"}},  # 50 in 2024 -> 6
        "david":   {"date_of_birth": {"ETERNITY": "1964-12-31"}},  # 60 in 2024 -> 7
    },
    "households": {
        "hh": {"parents": ["alice", "bob", "carol", "david"]}
    },
}

simulation = SimulationBuilder().build_from_entities(tax_benefit_system, situation)
result = simulation.calculate("vacation_weeks_entitlement", "2024")

for name, weeks in zip(["alice", "bob", "carol", "david"], result):
    print(f"{name}: {weeks:.0f} weeks")

## 6. Run the YAML test suite

In [ ]:
# The openfisca CLI test runner expects the variable + parameters to be
# importable via a country package. For an ad-hoc test inside the notebook,
# we drive the assertions manually against the situations declared in the YAML.
import yaml

with open(TESTS_DIR / "vacation_weeks_entitlement.yaml") as f:
    tests = yaml.safe_load(f)

for t in tests:
    period = str(t["period"])
    dob = str(t["input"]["date_of_birth"])
    expected = t["output"]["vacation_weeks_entitlement"]
    sit = {
        "persons": {"p": {"date_of_birth": {"ETERNITY": dob}}},
        "households": {"hh": {"parents": ["p"]}},
    }
    sim = SimulationBuilder().build_from_entities(tax_benefit_system, sit)
    got = float(sim.calculate("vacation_weeks_entitlement", period)[0])
    status = "OK" if got == expected else "FAIL"
    print(f"[{status}] {t['name']}: expected {expected}, got {got:.0f}")

## Notes on modelling choices

- **`definition_period = YEAR`** matches the legal unit ("pro Kalenderjahr").
- **Age semantics**: I compute `period.start.year - birth_year` rather than reading an `age` variable directly, because the law explicitly keys off the *calendar year in which the birthday falls*, not age at an instant. This matches "vom Beginn des Kalenderjahres an, in dem sie das X. Altersjahr vollenden".
- **`numpy.select` ordering** exploits first-match semantics: the four conditions are mutually exhaustive when evaluated in order, so explicit lower bounds on tiers 2-4 aren't needed.
- **`reference`** is a placeholder — fill in the actual Fedlex/ordinance URL once the source article is known (the provision reads like a company GAV or personnel ordinance clause).
- The parameter file uses `2000-01-01` as a placeholder start date; if the project has a known historical dated evolution (e.g. when the 7-week senior tier was introduced), add those dates instead.